# Momentum Strategy Code vvv

#### Import all modules, packages, tools to be used in the code.

In [ ]:
import numpy as np
import re
import pandas as pd
import pandas_datareader as pdr
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
import os
import re
import datetime 
from zipfile import ZipFile
import pandas_gbq
import janitor
import shutil
import glob
from google.cloud import bigquery
from google.oauth2 import service_account

In [ ]:
from scipy.stats import f_oneway

In [ ]:
# pd.set_option('display.max_columns', None)

In [ ]:
pd.set_option('display.max_rows', 30)
pd.set_option('display.min_rows', 30)

* Read in ES data to ES and change the column names to make sense. 
* Drop NA's to avoid issues, convert date data type, 
* sort date appropriately, add day number, set index to date.

In [ ]:
# ES = pd.read_csv('HistoricalData_ES.csv') #reads in ES data to ES
# ES['Close'] = ES['Close/Last'] #copies said column
# ES.drop(columns = ['Close/Last'], inplace = True) #drops 'columns' from dataframe, inplace makes it permanent
# ES = ES[['Date', 'Open', 'Close', 'High', 'Low', 'Volume']] #orders columns
# ES.dropna(inplace=True) #drops records that we can't really use 
# ES #prints top and bottom five records of dataframe

In [ ]:
ES = pd.read_csv('ES_Granular/ES_continuous_adjusted_5min.txt', names = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
ES.dropna(inplace=True)

# df2csv1 = 'ES'
# df2csv2 = 'ESw'
# df2csv3 = 'ESe'

MAs = list(range(1, 201))
# MAs2 = list(range(26, 51))
# MAs3 = list(range(51, 76))
# MAs4 = list(range(76, 101))
# MAs5 = list(range(101, 126))
# MAs6 = list(range(126, 151))
# MAs7 = list(range(151, 176))
# MAs8 = list(range(176, 201))
prix_met = 'Open'

In [ ]:
# You must define these before going further!!!

timeframe = '30min'
nperiods = '100ps'
# ES

In [ ]:
ES['Date']=pd.to_datetime(ES.Date) #converts Date columns to datetime format
ES = ES.sort_values(['Date'], ascending = True) #sorts by Date column so earliest is first
ES = ES.set_index('Date')

In [ ]:
# day = np.arange(1, len(ES) + 1) #creates day column
# ES['Day'] = day #adds day column to dataframe
#ES.reset_index(drop = True, inplace = True)

#ES

In [ ]:
# #adjust the volume ma here
# vma = 5
# ES['candle_average']=(ES['High']+ ES['Low']+ES['Open']+ES['Close'])/4 #creates average of our price metrics
# ES['vma']=ES['Volume'].rolling(vma).mean()#.shift() #creates volume moving average using vma as input
# #ES

In [ ]:
# Creating copies of the data set for the other two MA styles
ESw = ES.copy()
ESe = ES.copy()

In [ ]:
# These lines create 'movement' columns in each MA's data frame
ES['movement'] = ES['Open'].diff().shift(-1)
ESw['movement'] = ESw['Open'].diff().shift(-1)
ESe['movement'] = ESe['Open'].diff().shift(-1)

In [ ]:
ES.dropna(inplace=True)

In [ ]:
ES = ES[0:4999]

# Simple Moving Average Iteration

### These cells iterate through lists of moving average periods and creates a column for each.

In [ ]:
sma_s_names = []

for ma in MAs :
    
    ES[f"sma_s{ma}"] = ES[prix_met].rolling(ma).mean()
    sma_s_names.append(f"sma_s{ma}")

sma_c_names = []    
    
for ma in MAs :
    
    ES[f"sma_c{ma}"] = ES[prix_met].rolling(ma).mean()
    sma_c_names.append(f"sma_c{ma}")

In [ ]:
# Defragging ES
ES = ES.copy()
ES

In [ ]:
# Calculate the number of iterations required to split the DataFrame into chunks of 100 rows

table_size = 1000

num_iterations = len(ES) // table_size + 1
subset_filenames = []

# Initialize an empty dictionary to store the subsets
subset_dict = {}

# Loop over the DataFrame and create a new DataFrame with 100 rows in each iteration
for i in range(num_iterations):
    start_idx = i * table_size  # Calculate the starting index for the subset
    end_idx = (i + 1) * table_size  # Calculate the ending index for the subset
    subset_ES = ES.iloc[start_idx:end_idx]  # Subset the DataFrame using iloc
    
    # Add the subset DataFrame to the dictionary with a key that includes the iteration index
    subset_dict[f'subset_{i}'] = subset_ES
    
    filename = f'subset_{i}'
    subset_filenames.append(filename)
    
print(num_iterations)    
subset_filenames

In [ ]:
subset = 'subset_5'
subset_dict[subset]

In [ ]:
# print(ES.columns)
# print(sma_s_names)
# print(sma_c_names)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# The SMA signal generator
signal_cols_s = []
for sub in subset_filenames :
    for sname in sma_s_names :
        for cname in sma_c_names :
            # Long signal
            subset_dict[sub][f"{sname} {cname} signal"] = np.where(subset_dict[sub][sname] > subset_dict[sub][cname], 1, 0)
            # Short signal
            subset_dict[sub][f"{sname} {cname} signal"] = np.where(subset_dict[sub][sname] < subset_dict[sub][cname], -1, subset_dict[sub][f"{sname} {cname} signal"]) 
            subset_dict[sub].dropna(inplace=True) 
            signal_cols_s.append(f"{sname} {cname} signal")
#             print(f"{sub} {sname} {cname} signal column finished")
    print(f"all {sub} signal columns finished")


In [ ]:
# for sub in subset_filenames :    
#     subset_dict[sub]['movement'] = subset_dict[sub]['Open'].diff().shift(-1)

In [ ]:
# # The SMA signal generator
# signal_cols_s = []

# for sname in sma_s_names :
#     for cname in sma_c_names :
#         # Long signal
#         ES[f"{sname} {cname} signal"] = np.where(ES[sname] > ES[cname], 1, 0)
#         # Short signal
#         ES[f"{sname} {cname} signal"] = np.where(ES[sname] < ES[cname], -1, ES[f"{sname} {cname} signal"]) 
#         ES.dropna(inplace=True) 
#         signal_cols_s.append(f"{sname} {cname} signal")

# signal_cols_s

# ES = ES.copy()
# ES

In [ ]:
# ES.to_csv('ES_signals_30min_100ps.csv')
# print('ES with MAs and signals written to csv')
# ES = pd.read_csv('ES_signals_30min_100ps.csv')
# print(df2csv1, df2csv2, df2csv3, timeframe, nperiods)
# ES.to_csv(f'{df2csv}_signals_{timeframe}_{nperiods}.csv')

# Weighted Moving Average Iteration

### Defining functions for both the fast WMA and slow WMA. 

In [ ]:
data = ESw['Open']

def weighted_moving_average(data, period, rule):
    
    weights = []
    weights_calc = []
    
    # Calculating weights
    for i in range(period + 1) :
        pd.to_numeric(weights_calc.append(i))
    for i in weights_calc :
        weights.append(i/sum(weights_calc))   
    weights.pop(0)
    
    # Using weights to calculate the WMAs
    wma = []
    for i in range(len(data) - period + 1):
        wma.append(sum(data[i:i+period] * weights) / sum(weights))
    
    # Adding the WMAs to ES data frame
    wma = [float('nan')] * (period - 1) + wma
    ESw[f"wma{rule}{period}"] = wma    

In [ ]:
for ma in MAs :
    weighted_moving_average(data, ma, '_s')
    
for ma in MAs :
    weighted_moving_average(data, ma, '_c')    

In [ ]:
wma_s_names = []
for ma in MAs :
    wma_s_names.append(f"wma_s{ma}")

wma_c_names = []
for ma in MAs :
    wma_c_names.append(f"wma_c{ma}")

In [ ]:
print(wma_s_names)
print(wma_c_names)

In [ ]:
ESw = ESw.copy()
ESw

In [ ]:
# The WMA signal generator
signal_cols_w = []

for sname in wma_s_names :
    for cname in wma_c_names :
        # Long signal
        ESw[f"{sname} {cname} signal"] = np.where(ESw[sname] > ESw[cname], 1, 0) 
        # Short signal
        ESw[f"{sname} {cname} signal"] = np.where(ESw[sname] < ESw[cname], -1, ESw[f"{sname} {cname} signal"]) 
        ESw.dropna(inplace=True)
        signal_cols_w.append(f"{sname} {cname} signal")

In [ ]:
print(signal_cols_w)

In [ ]:
ESw = ESw.copy()
ESw

In [ ]:
ESw.to_csv('ESw_signals.csv')
print('ESw with MAs and signals written to csv')

# Exponential Moving Average

In [ ]:
def exponential_moving_average(data, column_name, period, rule):

    # Calculate the weighting multiplier
    alpha = 2 / (period + 1)

    # Calculate the initial EMA value using a simple moving average
    sma = data[column_name].rolling(window=period, min_periods=period).mean()
    ema_init = sma[period-1]
    
    # Calculate the EMA values for the remaining data
    ema_values = []
    for i in range(period, len(data)):
        ema = alpha * data[column_name][i] + (1 - alpha) * ema_init
        ema_init = ema
        ema_values.append(ema)
    
    # Adding the EMAs to the ES data frame
    ema_values = [float('nan')] * (period) + ema_values
    data[f"ema{rule}{period}"] = ema_values

In [ ]:
for ma in MAs :
    exponential_moving_average(ESe, 'Open', ma, '_s')

for ma in MAs :
    exponential_moving_average(ESe, 'Open', ma, '_c') 

In [ ]:
ESe = ESe.copy()
ESe

In [ ]:
ema_s_names = []
for period in MAs :
    ema_s_names.append(f"ema_s{period}")

ema_c_names = []
for period in MAs :
    ema_c_names.append(f"ema_c{period}")

In [ ]:
print(ema_s_names)
print(ema_c_names)

In [ ]:
# The EMA signal generator
signal_cols_e = []

for sname in ema_s_names :
    for cname in ema_c_names :
        # Long signal
        ESe[f"{sname} {cname} signal"] = np.where(ESe[sname] > ESe[cname], 1, 0)
        # Short signal
        ESe[f"{sname} {cname} signal"] = np.where(ESe[sname] < ESe[cname], -1, ESe[f"{sname} {cname} signal"]) 
        ESe.dropna(inplace=True)
        signal_cols_e.append(f"{sname} {cname} signal")

In [ ]:
ESe = ESe.copy()
ESe

# Profit Calculations

### The following code uses my own methods for calculating profit based on point accrual.

In [ ]:
#running this code more than once in the kernal screws up the affected columns; 
#need to move the signal down one row with shift() if we use Open to Open differences
#could also possibly shift point_profit

# for sig_col in signal_cols :
#     ES[sig_col] = ES[sig_col].shift()

In [ ]:
# profit_cols_s = []
# profit_cols_w = []
# profit_cols_e = []

# def point_profit_gen(df, sig_cols, prof_cols):
#         for sig_col in sig_cols :
#             df[f"{sig_col} point profit"] = df['movement']*df[sig_col]
#             prof_cols.append(f"{sig_col} point profit")

#         df.dropna(inplace=True)

In [ ]:
profit_cols_s = []
profit_cols_w = []
profit_cols_e = []

def point_profit_gen(subset_dict, sig_cols, prof_cols):
    for sub in subset_filenames :
        for sig_col in sig_cols :
            subset_dict[sub][f"{sig_col} point profit"] = subset_dict[sub]['movement']*subset_dict[sub][sig_col]
            prof_cols.append(f"{sig_col} point profit")
#             print(f"{sub} {sig_col} point profit calculation complete")
        subset_dict[sub].dropna(inplace=True)
        print(f"{sub} point profit columns complete")

In [ ]:
point_profit_gen(subset_dict, signal_cols_s, profit_cols_s)

In [ ]:
subset = 'subset_4'
subset_dict[subset]

In [ ]:
# point_profit_gen(ES, signal_cols_s, profit_cols_s)
# point_profit_gen(ESw, signal_cols_w, profit_cols_w)
# point_profit_gen(ESe, signal_cols_e, profit_cols_e)

In [ ]:
profit_cols_s = profit_cols_s.copy()
profit_cols_s

In [ ]:
profit_cols_w = profit_cols_w.copy()
profit_cols_w

In [ ]:
profit_cols_e = profit_cols_e.copy()
profit_cols_e

In [ ]:
ESe = ESe.copy()
ESe

In [ ]:
ESw = ESw.copy()
ESw

In [ ]:
ES = ES.copy()
ES

In [ ]:
# Number of e-mini contracts traded 
# (one micro is one tenth of an e-mini, a full contract is five e-minis)
# Full value: $250
# e-mini value: $50
# Micro value: $5

contracts = 1
point_value = 50

#buy/hold futures dollars
dollars_bh = ((ES[prix_met][-1])-(ES[prix_met][0]))*point_value*contracts 
dollars_bh1 = "${:.2f}".format(dollars_bh)

print('Buy and Hold Strategy Returns')
print(dollars_bh1)     

In [ ]:
# In this cell, I need to fill each list with zeros at an equal length to the subset_dict subsets combined.
# This can be defined above. Then I need to iterate through the subsets and add column totals over again until the 
# overall totals are calculated.

profit_totals_s = []
profit_totals_w = []
profit_totals_e = []

def calc_profit(data, prof_cols, prof_tots):
    for prof_col in prof_cols :
        prof_tots.append(data[prof_col].sum()*point_value*contracts)

        print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
        print("${:.2f}".format(data[prof_col].sum()*point_value*contracts))
        print()

In [ ]:
subset ='subset_4'
prof_col = 'sma_s2 sma_c11 signal point profit'
subset_dict[subset][prof_col].sum()*point_value*contracts

In [ ]:
subset_dict[prof_col].values().sum()*point_value*contracts

In [ ]:
calc_profit(subset_dict, profit_cols_s, profit_totals_s)

In [ ]:
# calc_profit(ES, profit_cols_s, profit_totals_s)
# calc_profit(ESw, profit_cols_w, profit_totals_w)
# calc_profit(ESe, profit_cols_e, profit_totals_e)

In [ ]:
profit_totals_s

In [ ]:
profit_totals_w

In [ ]:
profit_totals_e

In [ ]:
# Create column name list.
col_names =  ['strategy', 'profit']
  
# Create an empty dataframe with columns.
strategy_performances_s  = pd.DataFrame(columns = col_names)
strategy_performances_w = strategy_performances_s.copy()
strategy_performances_e = strategy_performances_s.copy()

In [ ]:
strategy_performances_s.strategy = signal_cols_s
strategy_performances_s.profit = profit_totals_s

strategy_performances_w.strategy = signal_cols_w
strategy_performances_w.profit = profit_totals_w

strategy_performances_e.strategy = signal_cols_e
strategy_performances_e.profit = profit_totals_e

In [ ]:
# strat_perfs_s = strategy_performances_s.sort_values(by=['profit'], ascending = False)
# strat_perfs_w = strategy_performances_w.sort_values(by=['profit'], ascending = False)
# strat_perfs_e = strategy_performances_e.sort_values(by=['profit'], ascending = False)

strat_perfs_s = strategy_performances_s
strat_perfs_w = strategy_performances_w
strat_perfs_e = strategy_performances_e

# This block modified to obtain unordered profit totals for correlation coefficient calculation

In [ ]:
strat_perfs_s

In [ ]:
strat_perfs_w

In [ ]:
strat_perfs_e

# Step

In [ ]:
strat_perfs_s[['sma_s', 'sma_c']] = strat_perfs_s['strategy'].str.split(' ', 1, expand=True)
strat_perfs_s['sma_s'] = strat_perfs_s['sma_s'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_s['sma_c'] = strat_perfs_s['sma_c'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_s

In [ ]:
strat_perfs_w[['wma_s', 'wma_c']] = strat_perfs_w['strategy'].str.split(' ', 1, expand=True)
strat_perfs_w['wma_s'] = strat_perfs_w['wma_s'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_w['wma_c'] = strat_perfs_w['wma_c'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_w

In [ ]:
strat_perfs_e[['ema_s', 'ema_c']] = strat_perfs_s['strategy'].str.split(' ', 1, expand=True)
strat_perfs_e['ema_s'] = strat_perfs_e['ema_s'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_e['ema_c'] = strat_perfs_e['ema_c'].apply(lambda s: re.sub(r"\D", "", s))
strat_perfs_e

In [ ]:
subset_sps = strat_perfs_s[strat_perfs_s['profit'] > dollars_bh]
subset_sps

In [ ]:
subset_spw = strat_perfs_w[strat_perfs_w['profit'] > dollars_bh]
subset_spw

In [ ]:
subset_spe = strat_perfs_e[strat_perfs_e['profit'] > dollars_bh]
subset_spe

In [ ]:
dollars_bh

In [ ]:
strat_perfs_s.dtypes

In [ ]:
to_num = ['sma_s','sma_c']

for col in to_num :
    strat_perfs_s[col] = pd.to_numeric(strat_perfs_s[col])

strat_perfs_s.dtypes

In [ ]:
to_num = ['wma_s','wma_c']

for col in to_num :
    strat_perfs_w[col] = pd.to_numeric(strat_perfs_w[col])

strat_perfs_w.dtypes

In [ ]:
to_num = ['ema_s','ema_c']

for col in to_num :
    strat_perfs_e[col] = pd.to_numeric(strat_perfs_e[col])

strat_perfs_e.dtypes

In [ ]:
strat_perfs_s

In [ ]:
def conv_split(strat_perfs_, signal, control):
    return strat_perfs_.loc[
        np.where(strat_perfs_[signal] < strat_perfs_[control], True, False)
    ]['profit'].tolist()

SMA_conv = conv_split(strat_perfs_s,'sma_s','sma_c')
WMA_conv = conv_split(strat_perfs_w,'wma_s','wma_c')
EMA_conv = conv_split(strat_perfs_e,'ema_s','ema_c')

def inv_split(strat_perfs_, signal, control):
    return strat_perfs_.loc[
        np.where(strat_perfs_[signal] > strat_perfs_[control], True, False)
    ]['profit'].tolist()

SMA_inv = inv_split(strat_perfs_s,'sma_s','sma_c')
WMA_inv = inv_split(strat_perfs_w,'wma_s','wma_c')
EMA_inv = inv_split(strat_perfs_e,'ema_s','ema_c')

In [ ]:
# Initialize an empty DataFrame
rule_split = pd.DataFrame()

# Call the conv_split function and assign the output to columns in the DataFrame
rule_split['SMA_conv'] = SMA_conv
rule_split['WMA_conv'] = WMA_conv
rule_split['EMA_conv'] = EMA_conv
# Call the inv_split function and assign the output to columns in the DataFrame
rule_split['SMA_inv'] = SMA_inv
rule_split['WMA_inv'] = WMA_inv
rule_split['EMA_inv'] = EMA_inv
# Display the resulting DataFrame
rule_split

In [ ]:
print(rule_split['SMA_conv'].mean())
print(rule_split['WMA_conv'].mean())
print(rule_split['EMA_conv'].mean())
print(rule_split['SMA_inv'].mean())
print(rule_split['WMA_inv'].mean())
print(rule_split['EMA_inv'].mean())

In [ ]:
from tabulate import tabulate

# Set up the table headers
headers1 = ["MA/Rule", "Mean Profit"]

# Populate the table rows
rows = [
    ["SMA_conv", rule_split['SMA_conv'].mean()],
    ["WMA_conv", rule_split['WMA_conv'].mean()],
    ["EMA_conv", rule_split['EMA_conv'].mean()],
    ["SMA_inv", rule_split['SMA_inv'].mean()],
    ["WMA_inv", rule_split['WMA_inv'].mean()],
    ["EMA_inv", rule_split['EMA_inv'].mean()],
]

# Print the table using tabulate
print(tabulate(rows, headers=headers1, tablefmt="fancy_grid"))

In [ ]:
rule_split_hsd = pd.DataFrame(columns=['profit', 'rule'])

for num in SMA_conv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'SMA_conv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row

for num in WMA_conv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'WMA_conv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row
    
for num in EMA_conv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'EMA_conv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row

In [ ]:
for num in SMA_inv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'SMA_inv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row

for num in WMA_inv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'WMA_inv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row
    
for num in EMA_inv:
    # create a new row for each iteration
    new_row = {'profit': num, 'rule': 'EMA_inv'}
    # set the values for the "profit" and "rule" columns separately
    rule_split_hsd.loc[len(rule_split_hsd)] = new_row

In [ ]:
rule_split_hsd

In [ ]:
rule_split.to_csv('rule_split.csv', index=False)

In [ ]:
rule_split_hsd.to_csv('rule_split_hsd.csv', index=False)

In [ ]:
# Perform the ANOVA
f_statistic, p_value = f_oneway(rule_split['SMA_conv'], rule_split['WMA_conv'], rule_split['EMA_conv'])

# Print the results
print('F-statistic:', f_statistic)
print('p-value:', p_value)

# Interpret the results
if p_value < 0.05:
    print('There is significant difference between at least two of the group means.')
else:
    print('There is no significant difference between any of the group means.')

In [ ]:
# Perform the ANOVA
f_statistic, p_value = f_oneway(rule_split['SMA_conv'], rule_split['WMA_conv'])

# Print the results
print('F-statistic:', f_statistic)
print('p-value:', p_value)

# Interpret the results
if p_value < 0.05:
    print('There is significant difference between at least two of the group means.')
else:
    print('There is no significant difference between any of the group means.')

In [ ]:
rule_split['EMA_conv'].mean() - rule_split['SMA_conv'].mean()

In [ ]:
# if pd.to_numeric(re.findall(r'\d+', lname)) > pd.to_numeric(re.findall(r'\d+', sname)) :

In [ ]:
color = 'white'

def heat_map_viz(title, strat_perf, _ma_s, _ma_c):
    
    plt.rcParams['xtick.color'] = color
    plt.rcParams['ytick.color'] = color
    plt.rcParams['axes.labelcolor'] = color
    plt.rcParams['axes.edgecolor'] = color
    plt.rcParams['axes.titlecolor'] = color
    plt.rcParams.update({'font.size': 15})
    plt.title(title)
    plt.rcParams['figure.figsize'] = 20,15 
    X = strat_perf[_ma_s]
    Y = strat_perf[_ma_c]
    Z = strat_perf['profit']
    data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
    data_pivoted = data.pivot("X", "Y", "Z")
    ax = sns.heatmap(data_pivoted)
    plt.xlabel('Control Period')
    plt.ylabel('Signal Period')
    ax.invert_yaxis()
    plt.show()

In [ ]:
heat_map_viz('SMA Strategy Profits', strat_perfs_s, 'sma_s', 'sma_c')

In [ ]:
heat_map_viz('WMA Strategy Profits', strat_perfs_w, 'wma_s', 'wma_c')

In [ ]:
heat_map_viz('EMA Strategy Profits', strat_perfs_e, 'ema_s', 'ema_c')

In [ ]:
# THINGS TO WORK ON
# 2. 
# 3.RUN THE MOST VOLUMINOUS CONTRACTS THROUGH THIS SCRIPT (NQ, GC, CL)
# 5.CALCULATE TOTAL TRANSACTION FEES
# 6.INVESTIGATE THAT SYSTEM RESULTS ARE CORRECT
# 7.FIND A WAY TO INCORPORATE TICK/VOLUME CANDLE DATA IN ANY WAY POSSIBLE
# 8.LEARN HOW TO USE A TRADING API
# 9.LEARN TO USE JAVASCRIPT/C# IF NECESSARY
# 11.CONSIDER ADDING THE COLUMNS AND ENRICHING THE DATA IN THE WAYS YOU DREAMT ABOUT
#    -DIFFERENCE BETWEEN SMA & LMA
#     -RELATIVE VOLUME OR DIFFERENCE BETWEEN VOLUME AND VMA
#      -MOVING AVERAGE RIBBON AND SUM OF DIFFERENCE
#       -RSI CALCULATION
#        -LOCAL HIGHS AND LOWS
#         -VWAP CALCULATION
#          -
# 12. CONSIDER CONCATENATING THE MOVING AVERAGE VALUES TO MAKE THE TABLE SMALLER
#    -YOU WILL HAVE TO FIGURE OUT HOW TO ACCESS THESE SUBSTRINGS FOR CALCULATING SIGNALS

### VVV Old Iterations Code

In [ ]:
# data = ESw['Open']

# def weighted_moving_average(data, period):
    
#     weights = []
#     weights_calc = []

#     for i in range(period + 1) :
#         pd.to_numeric(weights_calc.append(i))
#     for i in weights_calc :
#         weights.append(i/sum(weights_calc))   
#     weights.pop(0)
    
#     wma = []
#     for i in range(len(data) - period + 1):
#         wma.append(sum(data[i:i+period] * weights) / sum(weights))
    
#     wma = [float('nan')] * (period - 1) + wma
#     ESw[f"{period}_lwma"] = wma
    
# def weighted_moving_average_s(data, period):
    
#     weights = []
#     weights_calc = []

#     for i in range(period + 1) :
#         pd.to_numeric(weights_calc.append(i))
#     for i in weights_calc :
#         weights.append(i/sum(weights_calc))   
#     weights.pop(0)
    
#     wma = []
#     for i in range(len(data) - period + 1):
#         wma.append(sum(data[i:i+period] * weights) / sum(weights))
# #     return wma
#     wma = [float('nan')] * (period - 1) + wma
#     ESw[f"{period}_swma"] = wma

#### Applying the functions to the weighted data set and creating a defrag copy.

In [ ]:
# # Just for reference
# # LongMAs = list(range(1, 51)) # change this back to 51
# # SignalMAs = list(range(1, 51))
# # prix_met = 'Open'
# # prix_met_2 = 'Close'

# for ma in LongMAs :
#     weighted_moving_average(data, ma)

# for ma in SignalMAs :
#     weighted_moving_average_s(data, ma)    

In [ ]:
# l_wma_names

In [ ]:
# s_wma_names

In [ ]:
# profit_cols_s = []
# for sig_col in signal_cols_s :
#     ES[f"{sig_col} point profit"] = ES['movement']*ES[sig_col]
#     profit_cols_s.append(f"{sig_col} point profit")
    
# ES.dropna(inplace=True) #drops records that we can't really use

In [ ]:
# profit_cols_w = []
# for sig_col in signal_cols_w :
#     ESw[f"{sig_col} point profit"] = ESw['movement']*ESw[sig_col]
#     profit_cols_w.append(f"{sig_col} point profit")
    
# ESw.dropna(inplace=True) #drops records that we can't really use

In [ ]:
# profit_cols_e = []
# for sig_col in signal_cols_e :
#     ESe[f"{sig_col} point profit"] = ESe['movement']*ESe[sig_col]
#     profit_cols_e.append(f"{sig_col} point profit")
    
# ESe.dropna(inplace=True) #drops records that we can't really use

In [ ]:
# profit_totals_s = []
# for prof_col in profit_cols_s :
#     profit_totals_s.append(ES[prof_col].sum()*point_value*contracts)
    
#     print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
#     print("${:.2f}".format(ES[prof_col].sum()*point_value*contracts))
#     print()

In [ ]:
# profit_totals_w = []
# for prof_col in profit_cols_w :
#     profit_totals_w.append(ESw[prof_col].sum()*point_value*contracts)
    
#     print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
#     print("${:.2f}".format(ESw[prof_col].sum()*point_value*contracts))
#     print()

In [ ]:
# profit_totals_e = []
# for prof_col in profit_cols_e :
#     profit_totals_e.append(ESe[prof_col].sum()*point_value*contracts)
    
#     print('Returns for '+prof_col.replace(' signal point profit', '')+ ' Strategy')
#     print("${:.2f}".format(ESe[prof_col].sum()*point_value*contracts))
#     print()

In [ ]:
# plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
# X = strat_perfs_s.sma_s
# Y = strat_perfs_s.sma_c
# Z = strat_perfs_s.profit
# data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
# data_pivoted = data.pivot("X", "Y", "Z")
# ax = sns.heatmap(data_pivoted)
# plt.xlabel('Control Period')
# plt.ylabel('Signal Period')
# ax.invert_yaxis()
# plt.show()

In [ ]:
# plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
# X = strat_perfs_w.wma_s
# Y = strat_perfs_w.wma_c
# Z = strat_perfs_w.profit
# data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
# data_pivoted = data.pivot("X", "Y", "Z")
# ax = sns.heatmap(data_pivoted)
# plt.xlabel('Control Period')
# plt.ylabel('Signal Period')
# ax.invert_yaxis()
# plt.show()

In [ ]:
# plt.rcParams['figure.figsize'] = 20,15 #sets size of the plot
# X = strat_perfs_e.ema_s
# Y = strat_perfs_e.ema_c
# Z = strat_perfs_e.profit
# data = pd.DataFrame({'X': X, 'Y': Y, 'Z': Z})
# data_pivoted = data.pivot("X", "Y", "Z")
# ax = sns.heatmap(data_pivoted)
# plt.xlabel('Control Period')
# plt.ylabel('Signal Period')
# ax.invert_yaxis()
# plt.show()